In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
from pathlib import Path
# 
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
%%RecordEvent
passmark = 40

In [ ]:
### cell 0 ###

### cell 0 (optimized)
benchmark_name = "student-performance-in-exams"
# read into a cuDF-backed DataFrame
df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent
    / "input"
    / "StudentsPerformance.csv"
)
factor = 1000
# instead of concatenating a list of 1000 copies, use a single GPU take() with a repeated index
df = df.take(df.index.repeat(factor))
# df.take on a repeated index will automatically produce a new RangeIndex of the expanded length
df.info()

In [ ]:
### cell 1 ###

df.isna().sum()

In [ ]:
### cell 2 ###

df.head()

In [ ]:
### cell 3 ###

df.describe()

In [ ]:
### cell 4 ###

df.isnull().sum()

In [ ]:
### cell 5 ###

# Convert 'math score' to numeric on GPU
df['math score'] = pd.to_numeric(df['math score'], errors='coerce')

# Initialize all as 'P' then GPU‐native mask to set failing scores to 'F'
df['Math_PassStatus'] = 'P'
df['Math_PassStatus'] = df['Math_PassStatus'].mask(
    df['math score'] < passmark,
    'F'
)

# Count values (GPU)
df.Math_PassStatus.value_counts()

In [ ]:
### cell 6 ###

# Convert reading score to numeric (GPU)
df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')

# Initialize all as 'P' (on GPU)
df['Reading_PassStatus'] = 'P'

# Overwrite with 'F' where score is below passmark (GPU‐native boolean indexing)
df.loc[df['reading score'] < passmark, 'Reading_PassStatus'] = 'F'

# Count pass/fail (GPU)
df['Reading_PassStatus'].value_counts()

In [ ]:
### cell 7 ###

# Convert writing score to numeric on GPU
df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')

# Build a boolean mask for failures (writing score < passmark)
mask = df['writing score'] < passmark

# Initialize all as pass ('P') and then GPU‐accelerated mask failures to 'F'
df['Writing_PassStatus'] = 'P'
df['Writing_PassStatus'] = df['Writing_PassStatus'].mask(mask, 'F')

# Compute counts on GPU
df['Writing_PassStatus'].value_counts()

In [ ]:
### cell 8 ###

# Use vectorized boolean operations and masking to leverage GPU
mask = (
    (df['Math_PassStatus'] == 'F') |
    (df['Reading_PassStatus'] == 'F') |
    (df['Writing_PassStatus'] == 'F')
)

# Initialize all as 'P' then set 'F' where any fail occurs
df['OverAll_PassStatus'] = 'P'
df.loc[mask, 'OverAll_PassStatus'] = 'F'

# Compute the counts on GPU
df.OverAll_PassStatus.value_counts()

In [ ]:
### cell 9 ###

# Compute total marks via element-wise addition
cols = ['math score', 'reading score', 'writing score']
df['Total_Marks'] = df['math score'] + df['reading score'] + df['writing score']
# Compute percentage
df['Percentage'] = df['Total_Marks'] / 3

In [ ]:
### cell 10 ###

# Vectorized assignment for GPU execution
df['Grade'] = 'F'
mask = df['OverAll_PassStatus'] != 'F'
df.loc[mask & (df['Percentage'] >= 40), 'Grade'] = 'E'
df.loc[mask & (df['Percentage'] >= 50), 'Grade'] = 'D'
df.loc[mask & (df['Percentage'] >= 60), 'Grade'] = 'C'
df.loc[mask & (df['Percentage'] >= 70), 'Grade'] = 'B'
df.loc[mask & (df['Percentage'] >= 80), 'Grade'] = 'A'

df.Grade.value_counts()